### 연습 문제
- data 폴더 안에 'insurance.csv' 파일이 존재 
- 'charges' 종속 변수 
- torch의 다중 퍼셉트론을 이용하여 회귀형 모델을 생성 (epoch의 횟수는 300회 제한)
    - 종속변수도 2차원 데이터셋으로 변경 [ 1차 행렬 -> 2차 행렬 ]
    - 종속변수 데이터의 타입을 float32
- ML 모델로는 XGBoost을 이용하여 회귀 모델을 생성 
- R2 Score를 구해서 어떤 모델이 더 좋은 성능을 가지는가.
- XGBoost params 
    - n_estimator : [100, 200, 300]
    - max_depth = [3, 4, 5]
    - learning_rate : [0.01, 0.05, 0.1]
    - subsample : [0.8, 0.9, 1.0]

In [14]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

import torch 
import torch.nn as nn
import torch.optim as optim


In [3]:
df = pd.read_csv("../data/insurance.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


- age : 나이 
- sex : 성별
- bmi : 체질량수치
- children : 자녀의 수
- smoker : 흡연 여부
- region : 거주 지역
- charges : 의료비(target)

In [ ]:
df.head()

In [6]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
# 데이터 확인 
df['sex'].unique()

array(['female', 'male'], dtype=object)

In [8]:
df['smoker'].unique()

array(['yes', 'no'], dtype=object)

In [9]:
df['region'].unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [10]:
# 범주형 데이터들을 더미화 
df = pd.get_dummies(df, columns = ['sex', 'smoker', 'region'], drop_first = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               1338 non-null   int64  
 1   bmi               1338 non-null   float64
 2   children          1338 non-null   int64  
 3   charges           1338 non-null   float64
 4   sex_male          1338 non-null   bool   
 5   smoker_yes        1338 non-null   bool   
 6   region_northwest  1338 non-null   bool   
 7   region_southeast  1338 non-null   bool   
 8   region_southwest  1338 non-null   bool   
dtypes: bool(5), float64(2), int64(2)
memory usage: 48.5 KB


In [11]:
# 독립, 종속 데이터 분리 
x = df.drop('charges', axis = 1)
y = df['charges']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42
)

In [16]:
X_train.describe()

,age,bmi,children
count,1070.000000,1070.000000,1070.000000
mean,39.357009,30.560397,1.107477
std,14.073960,6.043386,1.215983
min,18.000000,15.960000,0.000000
25%,27.000000,26.205000,0.000000
50%,39.500000,30.210000,1.000000
75%,51.000000,34.496250,2.000000
max,64.000000,53.130000,5.000000


In [17]:
# 딥러닝 다중 퍼셉트론 

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Tensor로 변환 
X_train_tensor = torch.tensor(X_train_sc, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)

In [22]:
# 1차행렬의 데이터를 2차 행렬로 변환 
y_train_tensor =  torch.tensor(y_train.values.reshape(-1, 1), dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

print(y_train_tensor.shape)

torch.Size([1070, 1])


In [31]:
# Tensor에서 에서 제공하는 데이터의 구조를 바꾸는 방식 
y_train_tensor2 = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(-1)
y_test_tensor2 = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(-1)

In [ ]:
# view()함수는 numpy의 reshape()과 유사한 기능을 하는 함수입니다.
torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

tensor([[ 9193.8389],
        [ 8534.6719],
        [27117.9941],
        ...,
        [11931.1250],
        [46113.5117],
        [10214.6357]])

In [35]:
class Reg_Model(nn.Module):
    def __init__(self, _dim):
        super(Reg_Model, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(_dim, 64), 
            nn.ReLU(), 
            nn.Dropout(0.2), 
            nn.Linear(64, 32), 
            nn.ReLU(), 
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.model(x)

In [36]:
model = Reg_Model(X_train_tensor.shape[1])

In [37]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

In [38]:
epochs = 300

for epoch in range(epochs):
    # 모델의 예측값
    pred = model(X_train_tensor)
    # 예측값과 실제값의 차이를 계산
    loss = criterion(pred, y_train_tensor)
    # 기울기를 초기화 
    optimizer.zero_grad()
    # 역전파 계산 ( 자동 미분을 통해서 기울기의 방향을 알려준다. )
    loss.backward()
    # 가중치 업데이트(가중치를 역전파의 계산 방향으로 이동)
    optimizer.step()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {round(loss.item(), 6)}")

Epoch 30/300, Loss: 320740768.0
Epoch 60/300, Loss: 301979232.0
Epoch 90/300, Loss: 232271184.0
Epoch 120/300, Loss: 121906496.0
Epoch 150/300, Loss: 64388860.0
Epoch 180/300, Loss: 50906172.0
Epoch 210/300, Loss: 46185096.0
Epoch 240/300, Loss: 43480848.0
Epoch 270/300, Loss: 41531836.0
Epoch 300/300, Loss: 42654900.0
